### OpenWeather API

- Create an account [here](https://openweathermap.org/). You will use their API to get weather forecast data
- After you have created the free account, request an API key. The key enables you to connect to their API and pull weather data.
- Helpful video to understand API calls with Python [here](https://www.youtube.com/watch?v=_jBeFdqnNAU)

In [0]:
# Execute this cell
import pandas as pd
import numpy as np
import requests

# this cell includes the dataframes you'll use later
# creating the pandas dataframes
weather_error_logs = pd.DataFrame(columns=['ErrorCode','ErrorMessage','GivenCity','GivenLatitude','GivenLongtitude'])


### Weather Function
- Create a function which will make requests to the OpenWeather [Current Weather Data API](https://openweathermap.org/current) to get weather information using the latitude and longitude coordinates for a city. 
- Parse the response from the API request and extract the following information about a city: <code>id,name,country,main description,temperature,minimum temperature,maximum temperature,feels_like,humidity and wind speed<code>.
- The function should return a dictionary of the format:
  ```
    {
      "id": 802,
      "city": "Skopje",
      "latitude": 42,
      "longtitude": 21.4333,
      "country": 25,
      "description": "scattered clouds",
      "temp":290.97,
      "temp_min": 293.97,
      "temp_max": 293.97,
      "feels_like": 293.23,
      "humidity": 43,
      "wind_speed": 2.06   
     }
  ```
 - Design the function in a way that it should work for any given pair of values for latittude and longitute coordinates.

In [0]:
# insert here your API key
weather_api_key = 'd975b4c1b2ac3cddbdec76dcedec6a5f' 

# example of latittude,longtitude values for Skopje
city,lat,lon = 'Skopje', 42, 21.4333

def get_weather_data(city,lat,long,weather_api_key):
    print('Weather function')
    # write your code here
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={long}&appid={weather_api_key}"
    response = requests.get(url).json()
   
    if 'cod' in response and response['cod'] != 200:
        weather_error_logs.loc[len(weather_error_logs)] = {
            'ErrorCode': response.get('cod', 'unknown'),
            'ErrorMessage': response.get('message', 'Unknown error'),
            'GivenCity': city,
            'GivenLatitude': lat,
            'GivenLongtitude': long
        }
        return None
    
    if response['name'] != city:
        weather_error_logs.loc[len(weather_error_logs)] = {
            'ErrorCode': '404',
            'ErrorMessage': 'City not found',
            'GivenCity': city,
            'GivenLatitude': lat,
            'GivenLongtitude': long
        }
        return None
    id = response['weather'][0]['id']
    city = response['name']
    latitude = response['coord']['lat']
    longtitude = response['coord']['lon']
    country = response['sys']['country']
    description = response['weather'][0]['description']
    temp = response['main']['temp']
    temp_min = response['main']['temp_min']
    temp_max = response['main']['temp_max']
    feels_like = response['main']['feels_like']
    humidity = response['main']['humidity']
    wind_speed = response['wind']['speed']
    
    
    return{
        'id': id,
        'city': city,
        'latitude': latitude,
        'longtitude': longtitude,
        'country': country,
        'description': description,
        'temp': temp,
        'temp_min': temp_min,
        'temp_max': temp_max,
        'feels_like': feels_like,
        'humidity': humidity,
        'wind_speed': wind_speed
    }
    
    # The OpenWeatherAPI returns the temperature by default in Kelvin. To convert kelvin into celcius use the following formula: temp_celsius = Kelvin - 273.15


# calling the function
get_weather_data(city,lat,lon,weather_api_key)

Weather function


{'id': 500,
 'city': 'Skopje',
 'latitude': 42,
 'longtitude': 21.4333,
 'country': 'MK',
 'description': 'light rain',
 'temp': 298.45,
 'temp_min': 298.45,
 'temp_max': 298.45,
 'feels_like': 298.65,
 'humidity': 62,
 'wind_speed': 1.91}

In [0]:
# Run this cell to check your function
city,lat,lon = 'Skopje', 42, 21.4333

answer = get_weather_data(city,lat,lon,weather_api_key)

# checking if you are returning a dictionary
assert isinstance(answer,dict), "You are not returning a dictionary"

# checking if the returned dictionary contains all the right keys
weather_keys = [ "id","latitude","longtitude", "city","country","description","temp","temp_min","temp_max","feels_like","humidity","wind_speed"]
assert all(key in answer for key in weather_keys),"You are missing keys"


Weather function


#### Handle failures in weather function 

- A failure in this case as an example would be considered passing wrong values for latitude and longtitude. For this reason, modify the weather function you developed before to gracefully fail if wrong values for latitude and longtitude are passed. 
- The function should capture the response error message from the API call. In these cases, record this information as a new entry into the dataframe **weather_error_logs**. Explore the structure of this dataframe to see what information you need to capture and what columns you need to populate. If required, process the response further to populate the columns accordingly.

In [0]:
# clearing out the weather errors dataframe
weather_error_logs.drop(weather_error_logs.index,inplace=True)

# example of wrong values of latitude and longtitude 
city,lat,lon = 'Skopje',7955, 21.4333

# calling the function
get_weather_data(city,lat,lon,weather_api_key)

# Display first few rows of the weather_error_logs dataframe
weather_error_logs.head()

Weather function


,ErrorCode,ErrorMessage,GivenCity,GivenLatitude,GivenLongtitude
0,400,wrong latitude,Skopje,7955,21.4333


#### Get weather forecast for new locations
Below you are given a pandas dataframe **weather_df** holding data for new cities and their latitude and longtitude values. 
- For each city in the dataframe, by using the function you developed before, get the weather information and add it to the dataframe. Explore the structure of the dataframe and populate the columns accordingly.

In [0]:
# creating the pandas dataframes
weather_df = pd.DataFrame(
    data = np.array([["Skopje",42,21.4333], ["Tetovo",7958,21.4333],
                     ["London",51.5085,-0.1257],["Tokyo",35.6895,139.6917],
                    ["New York",40.7143,-74.006],["Berlin",52.5244,13.4105]]),
    columns=['city','latitude','longtitude'])


    
print('Weather Dataframe Before:\n',weather_df.head())

# Write your code here 
for col in ["id","country","description","temp","temp_min","temp_max","feels_like","humidity","wind_speed"]:
    weather_df[col] = None

for index, row in weather_df.iterrows():
    city = row['city']
    lat = float(row['latitude'])
    lon = float(row['longtitude'])
    
    weather_data = get_weather_data(city, lat, lon, weather_api_key)
    
    if weather_data is not None:
        for col in ["id","country","description","temp","temp_min","temp_max","feels_like","humidity","wind_speed"]:
            weather_df.at[index, col] = weather_data[col]  

print('Weather Dataframe After:\n' ,weather_df.head())


Weather Dataframe Before:
        city latitude longtitude
0    Skopje       42    21.4333
1    Tetovo     7958    21.4333
2    London  51.5085    -0.1257
3     Tokyo  35.6895   139.6917
4  New York  40.7143    -74.006
Weather function
Weather function
Weather function
Weather function
Weather function
Weather function
Weather Dataframe After:
        city latitude longtitude    id  ... temp_max feels_like humidity wind_speed
0    Skopje       42    21.4333   500  ...   298.45     298.65       62       1.91
1    Tetovo     7958    21.4333  None  ...     None       None     None       None
2    London  51.5085    -0.1257   803  ...    297.2     295.88       51       3.13
3     Tokyo  35.6895   139.6917   800  ...   297.39     297.42       90       0.99
4  New York  40.7143    -74.006   803  ...   305.51     306.37       49       2.68

[5 rows x 12 columns]


**Save the dataframes as tables**

- First, convert the two pandas dataframes `weather_df, weather_error_logs` into Spark dataframes. This [guide](https://docs.databricks.com/pandas/pyspark-pandas-conversion.html) shows how you can achieve that.
- Save the newly created `weather_df, weather_error_logs` Spark DataFrames as managed tables using `.write.mode("overwrite").saveAsTable()`.

> **💡 Tip:** When converting a Pandas DataFrame to a Spark DataFrame, make sure each column has a consistent data type (e.g., all numeric or all string). Mixed types in a column will cause an Arrow conversion error. Use `pd.to_numeric()` to cast columns that should be numeric.

In [0]:
# Write your code here

# Convert Pandas DataFrame to Spark DataFrame
weather_df_spark = spark.createDataFrame(weather_df)
weather_error_logs_spark = spark.createDataFrame(weather_error_logs)

# Save Spark DataFrames as managed tables
weather_df_spark.write.mode("overwrite").saveAsTable("weather_data")
weather_error_logs_spark.write.mode("overwrite").saveAsTable("weather_error_logs")

In [0]:
%sql
-- Displaying the tables
--select * from `weather_data`;
select * from `weather_error_logs`;

ErrorCode,ErrorMessage,GivenCity,GivenLatitude,GivenLongtitude
400,wrong latitude,Skopje,7955.0,21.4333
400,wrong latitude,Tetovo,7958.0,21.4333
404,City not found,Berlin,52.5244,13.4105
